In [153]:
import os
from pathlib import Path

import psycopg2
import pandas as pd
from datetime import datetime

# Load environment variable from app/db/postgres/.env
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.partition('=')
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value

PG_USER = os.getenv('DB_USER')
PG_PASSWORD = os.getenv('DB_PASSWORD')
PG_DB = os.getenv('DB_NAME')
PG_HOST = os.getenv('DB_HOST', 'localhost')
PG_PORT = os.getenv('DB_PORT', '5432')

print('Connecting to Postgres with:')
print('  host=', PG_HOST)
print('  port=', PG_PORT)
print('  db  =', PG_DB)
print('  user=', PG_USER)

conn = psycopg2.connect(
    dbname=PG_DB,
    user=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
)
cursor = conn.cursor()

cursor.execute('SELECT current_database(), version();')
print('Current database and version:')
print(cursor.fetchone())


Connecting to Postgres with:
  host= 136.116.194.34
  port= 3893
  db  = csdltmdv
  user= admin
Current database and version:
('csdltmdv', 'PostgreSQL 15.17 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit')


In [113]:
print('\nCurrent public tables:')
cursor.execute(
    '''
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
    '''
)
for (name,) in cursor.fetchall():
    print(name)


Current public tables:
currency
forecast_provider
indicator
indicator_data
price_forecast
price_history
price_latest
price_source
product
product_group
region
stock_index_master
stock_index_price
unit


In [154]:
# Drop tables if they exist, in order of dependencies to avoid FK violations.
drop_tables = [
    'price_history',
    # 'price_forecast',
    # 'indicator_data',
    # 'stock_index_price',
    # 'product',
]

for tbl in drop_tables:
    try:
        cursor.execute(f'DROP TABLE IF EXISTS {tbl} CASCADE;')
        print(f"Dropped table {tbl} (if existed).")
    except Exception as exc:
        print(f"Failed to drop table {tbl}: {exc}")
conn.commit()

Dropped table price_history (if existed).


### `currency`

In [35]:
# Load units from CSV (same file as in db_prep.ipynb)
currency = pd.read_csv('csv/currency.csv', encoding='utf-8', sep=';')
if 'id' in currency.columns:
    currency = currency.drop(columns=['id'])
if 'created_at' in currency.columns:
    currency = currency.drop(columns=['created_at'])
currency.head()

,currency_code,name
0,USD,US Dollar
1,VND,Vietnamese Dong


In [ ]:
try:
    for _, row in currency.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO currency (currency_code, name)
                VALUES (%s, %s);
                ''',
                (
                    row['currency_code'],
                    row['name']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted currency into the 'currency' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

Inserted units into the 'currency' table


In [38]:
cursor.execute("SELECT * FROM currency;")
all_currency = cursor.fetchall()
currency_db_df = pd.DataFrame(all_currency, columns=[desc[0] for desc in cursor.description])
currency_db_df.head(5)

,currency_code,name
0,USD,US Dollar
1,VND,Vietnamese Dong


### `forecast_provider`

In [ ]:
forecast_provider = pd.read_csv('csv/forecast_provider.csv', encoding='utf-8', sep=';')
if 'id' in forecast_provider.columns:
    forecast_provider = forecast_provider.drop(columns=['id'])
if 'created_at' in forecast_provider.columns:
    forecast_provider = forecast_provider.drop(columns=['created_at'])
forecast_provider.head()

,forecast_provider_code,forecast_provider_name
0,WM,Wood Mackenzie
1,EIA,U.S Energy Information Administration
2,REUTERS,Thomson Reuters
3,VPI,Vietnam Petroleum Institute


In [42]:
try:
    for _, row in forecast_provider.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO forecast_provider (forecast_provider_code, forecast_provider_name)
                VALUES (%s, %s);
                ''',
                (
                    row['forecast_provider_code'],
                    row['forecast_provider_name']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted forecast_provider into the 'forecast_provider' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

Inserted forecast_provider into the 'forecast_provider' table


In [43]:
cursor.execute("SELECT * FROM forecast_provider;")
forecast_provider = cursor.fetchall()
forecast_provider_db_df = pd.DataFrame(forecast_provider, columns=[desc[0] for desc in cursor.description])
forecast_provider_db_df.head(5)

,forecast_provider_code,forecast_provider_name
0,WM,Wood Mackenzie
1,EIA,U.S Energy Information Administration
2,REUTERS,Thomson Reuters
3,VPI,Vietnam Petroleum Institute


### `indicator`

In [44]:
indicator = pd.read_csv('csv/indicator.csv', encoding='utf-8', sep=';')
if 'id' in indicator.columns:
    indicator = indicator.drop(columns=['id'])
if 'created_at' in indicator.columns:
    indicator = indicator.drop(columns=['created_at'])
indicator.head()

,indicator_code,name,category
0,PMI,Manufacturing PMI,MACRO
1,GDP_GROWTH,GDP Growth Forecast,MACRO
2,INFLATION,Inflation Forecast,MACRO
3,CPI,Consumer Price Index,MACRO


In [45]:
try:
    for _, row in indicator.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO indicator (indicator_code, name, category)
                VALUES (%s, %s, %s);
                ''',
                (
                    row['indicator_code'],
                    row['name'],
                    row['category']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted indicator into the 'indicator' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

Inserted indicator into the 'indicator' table


In [46]:
cursor.execute("SELECT * FROM indicator;")
indicator = cursor.fetchall()
indicator_db_df = pd.DataFrame(indicator, columns=[desc[0] for desc in cursor.description])
indicator_db_df.head(5)

,indicator_code,name,category
0,PMI,Manufacturing PMI,MACRO
1,GDP_GROWTH,GDP Growth Forecast,MACRO
2,INFLATION,Inflation Forecast,MACRO
3,CPI,Consumer Price Index,MACRO


### `price_source`

In [47]:
price_source = pd.read_csv('csv/price_source.csv', encoding='utf-8', sep=';')
if 'id' in price_source.columns:
    price_source = price_source.drop(columns=['id'])
if 'created_at' in price_source.columns:
    price_source = price_source.drop(columns=['created_at'])
price_source.head()

,source_code,name
0,DOM,Domestic
1,INTL,International


In [60]:
try:
    for _, row in price_source.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO price_source (source_code, name)
                VALUES (%s, %s);
                ''',
                (
                    row['source_code'],
                    row['name']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted price_source into the 'price_source' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM price_source;")
price_source = cursor.fetchall()
price_source_db_df = pd.DataFrame(price_source, columns=[desc[0] for desc in cursor.description])
price_source_db_df.head(5)

Inserted price_source into the 'price_source' table


,source_code,name
0,DOM,Domestic
1,INTL,International


### `product`

In [83]:
product = pd.read_csv('csv/product.csv', encoding='utf-8', sep=';')
if 'id' in product.columns:
    product = product.drop(columns=['id'])
product.head(3)

,prod_code,prod_name,group_code
0,MOGAS95,Mogas 95,FUEL
1,MOGAS92,Mogas 92,FUEL
2,DO005S,Diesel 0.05% S,FUEL


In [84]:
try:
    for _, row in product.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO product (prod_code, prod_name, group_code)
                VALUES (%s, %s, %s);
                ''',
                (
                    row['prod_code'],
                    row['prod_name'],
                    row['group_code']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted product into the 'product' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")
    
cursor.execute("SELECT * FROM product;")
product = cursor.fetchall()
product_db_df = pd.DataFrame(product, columns=[desc[0] for desc in cursor.description])
product_db_df.head(5)

Inserted product into the 'product' table


,id,prod_code,prod_name,group_code
0,1,MOGAS95,Mogas 95,FUEL
1,2,MOGAS92,Mogas 92,FUEL
2,3,DO005S,Diesel 0.05% S,FUEL
3,4,FO180CST,Fuel Oil 180 CST,FUEL
4,5,MARINE5,Marine Fuel 5% S,FUEL


### `product_group`

In [ ]:
product_group = pd.read_csv('csv/product_group.csv', encoding='utf-8', sep=';')
if 'id' in product_group.columns:
    product_group = product_group.drop(columns=['id'])
if 'created_at' in product_group.columns:
    product_group = product_group.drop(columns=['created_at'])
product_group.head(3)

,group_code,group_name
0,FUEL,Refined Petroleum Products
1,CRUDE,Crude Oil
2,AGRI,Agricultural Products


In [64]:
try:
    for _, row in product_group.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO product_group (group_code, group_name)
                VALUES (%s, %s);
                ''',
                (
                    row['group_code'],
                    row['group_name']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted product_group into the 'product_group' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM product_group;")
product_group = cursor.fetchall()
product_group_db_df = pd.DataFrame(product_group, columns=[desc[0] for desc in cursor.description])
product_group_db_df.head(5)

Inserted product_group into the 'product_group' table


,group_code,group_name
0,FUEL,Refined Petroleum Products
1,CRUDE,Crude Oil
2,AGRI,Agricultural Products
3,FERT,Fertilizers


### `region`

In [52]:
region = pd.read_csv('csv/region.csv', encoding='utf-8', sep=';')
if 'id' in region.columns:
    region = region.drop(columns=['id'])
if 'created_at' in region.columns:
    region = region.drop(columns=['created_at'])
region.head(3)

,region_code,region_name
0,RUS,Russia
1,USA,United States
2,CHN,China


In [67]:
try:
    for _, row in region.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO region (region_code, region_name)
                VALUES (%s, %s);
                ''',
                (
                    row['region_code'],
                    row['region_name']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted region into the 'region' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM region;")
region = cursor.fetchall()
region_db_df = pd.DataFrame(region, columns=[desc[0] for desc in cursor.description])
region_db_df.head(5)

Inserted region into the 'region' table


,region_code,region_name
0,RUS,Russia
1,USA,United States
2,CHN,China
3,GBR,United Kingdom
4,DEU,Germany


### `stock_index_master`

In [53]:
stock_index_master = pd.read_csv('csv/stock_index_master.csv', encoding='utf-8', sep=';')
if 'id' in stock_index_master.columns:
    stock_index_master = stock_index_master.drop(columns=['id'])
if 'created_at' in stock_index_master.columns:
    stock_index_master = stock_index_master.drop(columns=['created_at'])
stock_index_master.head(3)

,code,name,market
0,DJI,Dow Jones,USA
1,COMP,NASDAQ COMPOSITE,USA
2,SPX,S&P 500,USA


In [68]:
try:
    for _, row in stock_index_master.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO stock_index_master (code, name, market)
                VALUES (%s, %s, %s);
                ''',
                (
                    row['code'],
                    row['name'],
                    row['market']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted stock_index_master into the 'stock_index_master' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM stock_index_master;")
stock_index_master = cursor.fetchall()
stock_index_master_db_df = pd.DataFrame(stock_index_master, columns=[desc[0] for desc in cursor.description])
stock_index_master_db_df.head(5)

Inserted stock_index_master into the 'stock_index_master' table


,code,name,market
0,DJI,Dow Jones,USA
1,COMP,NASDAQ COMPOSITE,USA
2,SPX,S&P 500,USA


### `unit`

In [54]:
unit = pd.read_csv('csv/unit.csv', encoding='utf-8', sep=';')
if 'id' in unit.columns:
    unit = unit.drop(columns=['id'])
if 'created_at' in unit.columns:
    unit = unit.drop(columns=['created_at'])
unit.head(3)

,unit_code,name
0,LITER,Liter
1,TON,Ton
2,BBL,Barrel


In [69]:
try:
    for _, row in unit.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO unit (unit_code, name)
                VALUES (%s, %s);
                ''',
                (
                    row['unit_code'],
                    row['name']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted unit into the 'unit' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM unit;")
unit = cursor.fetchall()
unit_db_df = pd.DataFrame(unit, columns=[desc[0] for desc in cursor.description])
unit_db_df.head(5)

Inserted unit into the 'unit' table


,unit_code,name
0,LITER,Liter
1,TON,Ton
2,BBL,Barrel
3,KG,Kilogam


### `stock_index_price`

In [85]:
stock_index_price = pd.read_csv('csv/stock_index_price.csv', encoding='utf-8', sep=';')
if 'id' in stock_index_price.columns:
    stock_index_price = stock_index_price.drop(columns=['id'])
stock_index_price['date'] = pd.to_datetime(stock_index_price['date'], format='%d/%m/%y', errors='coerce')
stock_index_price.head(3)


,code,date,close,currency
0,DJI,2026-03-25,46429,USD
1,COMP,2026-03-25,21929,USD
2,SPX,2026-03-25,6591,USD


In [86]:
try:
    for _, row in stock_index_price.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO stock_index_price (code, date, close, currency)
                VALUES (%s, %s, %s, %s);
                ''',
                (
                    row['code'],
                    row['date'],
                    row['close'],
                    row['currency']
                ),
                # If using SERIAL or GENERATED ALWAYS AS IDENTITY for 'id', we do not insert id explicitly; it is auto-incremented.
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted stock_index_price into the 'stock_index_price' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM stock_index_price;")
stock_index_price = cursor.fetchall()
stock_index_price_db_df = pd.DataFrame(stock_index_price, columns=[desc[0] for desc in cursor.description])
stock_index_price_db_df.head(5)

Inserted stock_index_price into the 'stock_index_price' table


,id,code,date,close,currency
0,1,DJI,2026-03-25,46429.0000,USD
1,2,COMP,2026-03-25,21929.0000,USD
2,3,SPX,2026-03-25,6591.0000,USD
3,4,DJI,2026-03-24,46124.0000,USD
4,5,COMP,2026-03-24,21761.0000,USD


### `indicator_data`

In [130]:
indicator_data = pd.read_csv('csv/indicator_data.csv', encoding='utf-8', sep=';')
if 'id' in indicator_data.columns:
    indicator_data = indicator_data.drop(columns=['id'])
indicator_data['created_at'] = pd.to_datetime(indicator_data['created_at'], format='%d/%m/%y', errors='coerce')
indicator_data['period'] = pd.to_datetime(indicator_data['period'], format='%d/%m/%y', errors='coerce')
indicator_data['indicator_code'] = indicator_data['indicator_code'].astype(str).str.strip()
indicator_data['region_code'] = indicator_data['region_code'].astype(str).str.strip()
indicator_data.head(3)

,indicator_code,region_code,value,unit,period,period_type,created_at
0,PMI,RUS,49.4,NaN,2026-01-01,MONTHLY,2026-02-05
1,PMI,USA,52.4,NaN,2026-01-01,MONTHLY,2026-02-05
2,PMI,CHN,49.3,NaN,2026-01-01,MONTHLY,2026-02-05


In [131]:
try:
    for _, row in indicator_data.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO indicator_data (indicator_code, region_code, value, unit, period, period_type, created_at)
                VALUES (%s, %s, %s, %s, %s, %s, %s);
                ''',
                (
                    row['indicator_code'],
                    row['region_code'],
                    row['value'],
                    row['unit'],
                    row['period'],
                    row['period_type'],
                    row['created_at']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted indicator_data into the 'indicator_data' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM indicator_data;")
indicator_data = cursor.fetchall()
indicator_data_db_df = pd.DataFrame(indicator_data, columns=[desc[0] for desc in cursor.description])
indicator_data_db_df.head(5)

Inserted indicator_data into the 'indicator_data' table


,id,indicator_code,region_code,value,unit,period,period_type,created_at
0,1,PMI,RUS,49.4000,NaN,2026-01-01 00:00:00,MONTHLY,2026-02-05
1,2,PMI,USA,52.4000,NaN,2026-01-01 00:00:00,MONTHLY,2026-02-05
2,3,PMI,CHN,49.3000,NaN,2026-01-01 00:00:00,MONTHLY,2026-02-05
3,4,PMI,GBR,51.8000,NaN,2026-01-01 00:00:00,MONTHLY,2026-02-05
4,5,PMI,DEU,49.1000,NaN,2026-01-01 00:00:00,MONTHLY,2026-02-05


### `price_forecast`

In [122]:
price_forecast = pd.read_csv('csv/price_forecast.csv', encoding='utf-8', sep=';')
if 'id' in price_forecast.columns:
    price_forecast = price_forecast.drop(columns=['id'])
price_forecast['created_at'] = pd.to_datetime(price_forecast['created_at'], format='%d/%m/%y', errors='coerce')
price_forecast['forecast_period'] = pd.to_datetime(price_forecast['forecast_period'], format='%d/%m/%y', errors='coerce')
price_forecast['prod_code'] = price_forecast['prod_code'].astype(str).str.strip()
price_forecast.head(3)

,prod_code,forecast_price,currency,unit,forecast_period,forecast_provider,period_type,created_at
0,LPG,58.7,USD,BBL,2026-01-01,WM,QUARTERLY,2026-03-15
1,NAPHTHA,72.7,USD,BBL,2026-01-01,WM,QUARTERLY,2026-03-15
2,RON92,83.8,USD,BBL,2026-01-01,WM,QUARTERLY,2026-03-15


In [123]:
try:
    for _, row in price_forecast.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO price_forecast (prod_code, forecast_price, currency, unit, forecast_period, forecast_provider, period_type, created_at)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
                ''',
                (
                    row['prod_code'],
                    row['forecast_price'],
                    row['currency'],
                    row['unit'],
                    row['forecast_period'],
                    row['forecast_provider'],
                    row['period_type'],
                    row['created_at']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted price_forecast into the 'price_forecast' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM price_forecast;")
price_forecast = cursor.fetchall()
price_forecast_db_df = pd.DataFrame(price_forecast, columns=[desc[0] for desc in cursor.description])
price_forecast_db_df.head(5)

Inserted price_forecast into the 'price_forecast' table


,id,prod_code,forecast_price,currency,unit,forecast_period,forecast_provider,period_type,created_at
0,1,LPG,58.7000,USD,BBL,2026-01-01 00:00:00,WM,QUARTERLY,2026-03-15
1,2,NAPHTHA,72.7000,USD,BBL,2026-01-01 00:00:00,WM,QUARTERLY,2026-03-15
2,3,RON92,83.8000,USD,BBL,2026-01-01 00:00:00,WM,QUARTERLY,2026-03-15
3,4,RON95,85.6000,USD,BBL,2026-01-01 00:00:00,WM,QUARTERLY,2026-03-15
4,5,JET,100.8000,USD,BBL,2026-01-01 00:00:00,WM,QUARTERLY,2026-03-15


### `price_history`

In [155]:
price_history = pd.read_csv('csv/price_history.csv', encoding='utf-8', sep=';')
if 'id' in price_history.columns:
    price_history = price_history.drop(columns=['id'])
price_history['date'] = pd.to_datetime(price_history['date'], format='%d/%m/%y', errors='coerce')
price_history.head(3)

,prod_code,price,currency,unit,date,source
0,MOGAS95,163.93,USD,TON,2026-03-20,INTL
1,MOGAS92,150.61,USD,TON,2026-03-20,INTL
2,DO005S,222.02,USD,TON,2026-03-20,INTL


In [ ]:
cursor.execute("SELECT * FROM price_history;")
price_history = cursor.fetchall()
price_history_db_df = pd.DataFrame(price_history, columns=[desc[0] for desc in cursor.description])
price_history_db_df.head(5)

In [156]:
try:
    for _, row in price_history.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO price_history (prod_code, price, currency, unit, date, source)
                VALUES (%s, %s, %s, %s, %s, %s);
                ''',
                (
                    row['prod_code'],
                    row['price'],
                    row['currency'],
                    row['unit'],
                    row['date'],
                    row['source']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted price_history into the 'price_history' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM price_history;")
price_history = cursor.fetchall()
price_history_db_df = pd.DataFrame(price_history, columns=[desc[0] for desc in cursor.description])
price_history_db_df.head(5)

Inserted price_history into the 'price_history' table


,id,prod_code,price,currency,unit,date,source
0,1,MOGAS95,163.9300,USD,TON,2026-03-20,INTL
1,2,MOGAS92,150.6100,USD,TON,2026-03-20,INTL
2,3,DO005S,222.0200,USD,TON,2026-03-20,INTL
3,4,FO180CST,740.7100,USD,TON,2026-03-20,INTL
4,5,MARINE5,929.0900,USD,TON,2026-03-20,INTL


### `price_latest`

In [142]:
price_latest = pd.read_csv('csv/price_latest.csv', encoding='utf-8', sep=';')
if 'id' in price_latest.columns:
    price_latest = price_latest.drop(columns=['id'])
price_latest['last_updated_at'] = pd.to_datetime(price_latest['last_updated_at'], format='%d/%m/%y', errors='coerce')
price_latest.head(3)

,prod_code,price,currency_code,last_updated_at,value_change,percentage_change,change_direction
0,MOGAS95,135.60,USD,2026-03-25,0.105,1.214,increase
1,MOGAS92,131.05,USD,2026-03-25,0.787,1.377,decrease
2,DO005S,204.62,USD,2026-03-25,0.865,1.621,decrease


In [145]:
for _, row in price_latest.iterrows():
    print(row['prod_code'])
    break

MOGAS95


In [146]:
try:
    for _, row in price_latest.iterrows():
        try:
            cursor.execute("SAVEPOINT row_savepoint")
            cursor.execute(
                '''
                INSERT INTO price_latest (prod_code, price, currency_code, last_updated_at, value_change, percentage_change, change_direction)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (prod_code)
                DO UPDATE SET
                    price = EXCLUDED.price,
                    value_change = EXCLUDED.value_change,
                    percentage_change = EXCLUDED.percentage_change,
                    change_direction = EXCLUDED.change_direction,
                    last_updated_at = EXCLUDED.last_updated_at;
                ''',
                (
                    row['prod_code'],
                    row['price'],
                    row['currency_code'],
                    row['last_updated_at'],
                    row['value_change'],
                    row['percentage_change'],
                    row['change_direction']
                ),
            )
            cursor.execute("RELEASE SAVEPOINT row_savepoint")
        except Exception as row_exc:
            cursor.execute("ROLLBACK TO SAVEPOINT row_savepoint")
            print(f"Could not insert row {row.to_dict()}: {row_exc}")

    conn.commit()
    print("Inserted price_latest into the 'price_latest' table")
except Exception as exc:
    conn.rollback()
    print(f"Error occurred during batch insert, rolled back transaction: {exc}")

cursor.execute("SELECT * FROM price_latest;")
price_latest = cursor.fetchall()
price_latest_db_df = pd.DataFrame(price_latest, columns=[desc[0] for desc in cursor.description])
price_latest_db_df.head(5)

Inserted price_latest into the 'price_latest' table


,prod_code,price,currency_code,last_updated_at,value_change,percentage_change,change_direction
0,MOGAS95,135.6000,USD,2026-03-25,0.1050,1.21,increase
1,MOGAS92,131.0500,USD,2026-03-25,0.7870,1.38,decrease
2,DO005S,204.6200,USD,2026-03-25,0.8650,1.62,decrease
3,FO180CST,637.8300,USD,2026-03-25,0.4570,1.43,decrease
4,MARINE5,793.8100,USD,2026-03-25,0.5280,1.34,decrease
